# Gold Layer - Customer Segmentation Dimension Table

## Purpose
Segment customers by purchase behavior for business analytics and targeted marketing.

## Type
**Dimension Table** (materialized, customer attributes)

## Input
* **Source:** `big_data.silver.orders` (3.3M rows)
* **Source:** `big_data.silver.order_products` (33.8M rows)
* **Source:** `big_data.silver.products_enriched` (49.7K rows)

## Output
* **Target:** `big_data.gold.dt_customer_segmentation`
* **Rows:** 206K (one per customer)
* **Primary Key:** user_id
* **Columns:** user_id, total_orders, total_items, estimated_lifetime_value_usd, avg_days_between_orders, reordered_items, purchase_frequency_segment

## Transformations

### Step 1: Load and Aggregate Customer Data
* JOIN orders with order_products and products
* GROUP BY user_id to calculate:
  * total_orders, total_items
  * estimated_lifetime_value_usd
  * avg_days_between_orders
  * reordered_items

### Step 2: Create Segments
* Assign purchase_frequency_segment:
  * New (no prior orders)
  * Weekly (≤7 days between orders)
  * Biweekly (8-14 days)
  * Monthly (15-30 days)
  * Occasional (>30 days)
* Add `_gold_timestamp`

## Data Quality Validations

### Technical Validations
* Customer count > 200K
* NOT NULL on user_id, total_orders

### Business Validations
* All 5 segments present
* Lifetime values > 0

## Why Materialized?
* ✅ 206K rows (large dimension)
* ✅ Used frequently in dashboards
* ✅ Complex segmentation logic
* ✅ Joined by other queries

## Persistence
Only persists if **all validations pass**.

## Execution
Expected runtime: ~5-7 minutes.

In [0]:
%run ../UTILS/data_quality_checks

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Target table (dimension table with dt_ prefix)
target_table = "dt_customer_segmentation"

# Expected segments for validation
expected_segments = ["New", "Weekly", "Biweekly", "Monthly", "Occasional"]

# Expected metrics
expected_metrics = {
    "min_customers": 200_000
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Target: {gold_schema}.{target_table}")

In [0]:
print("Step 1: Loading and aggregating...")

orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")
products = spark.table(f"{silver_schema}.products_enriched")

enriched = orders.join(order_products, "order_id").join(products.select("product_id", "price_usd"), "product_id", "left")

customer_agg = enriched.groupBy("user_id").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.count("product_id").alias("total_items"),
    F.round(F.sum("price_usd"), 2).alias("estimated_lifetime_value_usd"),
    F.round(F.avg(F.when(F.col("days_since_prior_order").isNotNull(), F.col("days_since_prior_order"))), 2).alias("avg_days_between_orders"),
    F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
)

print(f"  Customers: {customer_agg.count():,}")

In [0]:
print("Step 2: Creating customer segments and lifecycle metrics...")

# Create customer segmentation with purchase frequency segments
customer_segmentation_gold = customer_agg \
    .withColumn(
        "purchase_frequency_segment",
        F.when(F.col("avg_days_between_orders").isNull(), "New")
         .when(F.col("avg_days_between_orders") <= 7, "Weekly")
         .when(F.col("avg_days_between_orders") <= 14, "Biweekly")
         .when(F.col("avg_days_between_orders") <= 30, "Monthly")
         .otherwise("Occasional")
    ) \
    .withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Customer segmentation: {customer_segmentation_gold.count():,} customers")

# Create overall lifecycle metrics
customer_lifecycle_gold = customer_segmentation_gold.agg(
    F.countDistinct("user_id").alias("total_customers"),
    F.round(F.avg("total_orders"), 2).alias("avg_orders_per_customer"),
    F.round(F.avg("estimated_lifetime_value_usd"), 2).alias("avg_customer_lifetime_value_usd")
).withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Lifecycle metrics: 1 aggregated row")

print("\nPreview - Customer Segmentation (Sample 5):")
customer_segmentation_gold.orderBy(F.desc("estimated_lifetime_value_usd")).show(5, truncate=False)

print("\nPreview - Customer Lifecycle Metrics:")
customer_lifecycle_gold.show(1, truncate=False, vertical=True)

In [0]:
print_validation_header("customer_segmentation - Technical Validations")

# Initialize validation flag
validation_passed_technical = True

# 1. Customer count check
customer_count = customer_segmentation_gold.count()
print(f"\nTotal customers: {customer_count:,}")
print(f"Expected: >= {expected_metrics['min_customers']:,}\n")

if customer_count >= expected_metrics["min_customers"]:
    status = "PASS"
    msg = f"Customer count ({customer_count:,}) >= {expected_metrics['min_customers']:,}"
else:
    status = "FAIL"
    msg = f"Customer count ({customer_count:,}) < {expected_metrics['min_customers']:,}"
    validation_passed_technical = False
print_check_result("CUSTOMER COUNT", status, msg)

# 2. NOT NULL checks
print("\n2. NOT NULL Validations:")
status, failed, msg = check_not_null(customer_segmentation_gold, ["user_id", "total_orders"])
print_check_result("NOT NULL (user_id, total_orders)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

# 3. Lifecycle metrics check
lifecycle_count = customer_lifecycle_gold.count()
if lifecycle_count == 1:
    status = "PASS"
    msg = "Lifecycle metrics: 1 aggregated row"
else:
    status = "FAIL"
    msg = f"Lifecycle metrics: {lifecycle_count} rows (expected 1)"
    validation_passed_technical = False
print_check_result("LIFECYCLE METRICS ROW COUNT", status, msg)

print("\n" + "="*60)
if validation_passed_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("customer_segmentation - Business Validations")

# Initialize business validation flag
validation_passed_business = True

# 1. All segments present
print("\n1. Business Rule - Segment Coverage:")
actual_segments = [r.purchase_frequency_segment for r in customer_segmentation_gold.select("purchase_frequency_segment").distinct().collect()]
missing_segments = set(expected_segments) - set(actual_segments)

if len(missing_segments) == 0:
    status = "PASS"
    msg = f"All {len(expected_segments)} segments present: {sorted(actual_segments)}"
else:
    status = "FAIL"
    msg = f"Missing segments: {missing_segments}"
    validation_passed_business = False
print_check_result("SEGMENT COVERAGE (5 segments)", status, msg)

# 2. Segment distribution
print("\n2. Business Rule - Segment Distribution:")
segment_dist = customer_segmentation_gold.groupBy("purchase_frequency_segment").count().orderBy(F.desc("count")).collect()
print("\n  Distribution:")
for seg in segment_dist:
    print(f"    - {seg['purchase_frequency_segment']}: {seg['count']:,} customers ({seg['count']/customer_count*100:.1f}%)")

# 3. Lifetime value validation
print("\n3. Business Rule - Lifetime Value:")
zero_value_customers = customer_segmentation_gold.filter(F.col("estimated_lifetime_value_usd") <= 0).count()

if zero_value_customers == 0:
    status = "PASS"
    msg = "All customers have positive lifetime value"
else:
    status = "FAIL"
    msg = f"{zero_value_customers} customers with non-positive lifetime value"
    validation_passed_business = False
print_check_result("LIFETIME VALUE > 0", status, msg, zero_value_customers)

print("\n" + "="*60)
if validation_passed_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
validation_passed = validation_passed_technical and validation_passed_business
print("\n" + "="*60)
print("OVERALL VALIDATION")
print(f"  Technical: {'PASSED' if validation_passed_technical else 'FAILED'}")
print(f"  Business:  {'PASSED' if validation_passed_business else 'FAILED'}")
print("="*60)

In [0]:
# Only persist if validation passed
if validation_passed:
    print("Persisting customer segmentation to Gold layer...\n")
    
    # Save dt_customer_segmentation
    customer_segmentation_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{target_table}")
    
    # Verify
    seg_count = spark.table(f"{gold_schema}.{target_table}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Customer segmentation table persisted to Gold layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {gold_schema}.{target_table}")
    print(f"  Rows: {seg_count:,}")
    print(f"  Type: Dimension table (customer attributes)")
    print(f"  Format: Delta")
    print(f"\nNext Step: Query for customer insights and segmentation analysis")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")